# Infrastructure & Construction Monitoring
**NB 18** | NEOM megaproject, Saudi Arabia | Sentinel-2 time-series | ChangeFormer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.change_detection.changeformer import ChangeDetection


In [ ]:
BBOX        = (36.50, 28.00, 37.00, 28.50)   # NEOM project site, Tabuk region
DATE_2021   = ('2021-01-01', '2021-06-30')    # Project start phase
DATE_2024   = ('2024-01-01', '2024-06-30')    # Current construction state
PROVIDERS   = ['planetary_computer']
BANDS       = ['B02','B03','B04','B08','B11']  # Include SWIR for construction material detection
DATA_DIR    = pathlib.Path('./results/nb18')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
r2021 = client.search(bbox=BBOX, date_range=DATE_2021, providers=PROVIDERS, cloud_cover_max=10)
r2024 = client.search(bbox=BBOX, date_range=DATE_2024, providers=PROVIDERS, cloud_cover_max=10)
print(f"2021: {len(r2021)} scenes | 2024: {len(r2024)} scenes")
d2021 = client.download(r2021[:1], output_dir=str(DATA_DIR/'2021'), bands=BANDS, post_process=['reproject:EPSG:32637','cog'])
d2024 = client.download(r2024[:1], output_dir=str(DATA_DIR/'2024'), bands=BANDS, post_process=['reproject:EPSG:32637','cog'])
s2021 = d2021[0].path if d2021 and d2021[0].success else None
s2024 = d2024[0].path if d2024 and d2024[0].success else None


In [ ]:
R2021 = DATA_DIR/'2021_ready.tif'; R2024 = DATA_DIR/'2024_ready.tif'
for scene, out, label in [(s2021, R2021,'2021'), (s2024, R2024,'2024')]:
    if scene:
        rep = client.validator.validate(str(scene), required_bands=len(BANDS))
        r   = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=BBOX,
            normalise='scale_factor', scale_factor=10000.0,
            model_type='change_detection', output_path=str(out))
        print(f"{label}: {r['steps']}  shape={r['shape']}")


In [ ]:
# NDBI on both years to track built-up expansion
import rasterio
idx_dir = DATA_DIR/'indices'; idx_dir.mkdir(exist_ok=True)
for year, path in [('2021', R2021), ('2024', R2024)]:
    if path.exists():
        ndbi = client.indices.ndbi(str(path), swir1_band=5, nir_band=4,
            output_path=str(idx_dir/f'ndbi_{year}.tif'))
        bsi  = client.indices.bsi(str(path),  blue_band=1, red_band=3, nir_band=4,
            output_path=str(idx_dir/f'bsi_{year}.tif'))
        with rasterio.open(ndbi) as src: a = src.read(1)
        print(f"NDBI {year}: built-up={( a>0.05).mean()*100:.1f}%  mean={a.mean():.3f}")


In [ ]:
CHANGE_MAP = DATA_DIR/'construction_change.tif'
PREDICTION = CHANGE_MAP
if R2021.exists() and R2024.exists():
    cd = ChangeDetection(model_variant='changeformer', in_channels=len(BANDS))
    cd.build()
    res = cd.detect(str(R2021), str(R2024), output_path=str(CHANGE_MAP))
    print(f"Construction activity: {res['change_pct']:.2f}% of study area")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=20, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=500.0, simplify_tolerance=2.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


**NB18 — Infrastructure** complete.
- Study area: NEOM Saudi Arabia
- Sensor: Sentinel-2 time-series
- Model: ChangeFormer
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG